# الفصل 9: من التكرار إلى الانتباه: آلية Attention التاريخية

> تخيّل أنك تُترجم فقرة كاملة من العربية إلى الإنجليزية، لكنك مُجبَر على قراءتها مرة واحدة، ثم إغلاق عينيك، وكتابة الترجمة بالكامل من الذاكرة دون العودة للنص الأصلي.

- **الجزء:** 3 — المحوّلات الأساسية
- **الفصل:** 9
- **المستوى:** متوسط
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 8: الشبكات المتكررة: RNN وLSTM وGRU
- **الوسوم:** Bahdanau، Luong، Seq2Seq، Attention

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch09-attention-history.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-03-transformers%2Fch09-attention-history.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-03-transformers/ch09-attention-history.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch09-attention-history.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch09-attention-history.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch09-attention-history.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** numpy, torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("numpy", "numpy"),
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch09-attention-history.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-03-transformers%2Fch09-attention-history.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-03-transformers/ch09-attention-history.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch09-attention-history.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch09-attention-history.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch09-attention-history.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> تخيّل أنك تُترجم فقرة كاملة من العربية إلى الإنجليزية، لكنك مُجبَر على قراءتها مرة واحدة، ثم إغلاق عينيك، وكتابة الترجمة بالكامل من الذاكرة دون العودة للنص الأصلي.

> **بيانات الفصل**
>
> - **الجزء:** 3 — المحوّلات الأساسية
> - **المستوى:** متوسط
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 8: الشبكات المتكررة: RNN وLSTM وGRU


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Luong Attention (2015) — تبسيط وتعميم
- فهم Bahdanau
- فهم Luong


## 1. لماذا يهمّك هذا الموضوع؟

تخيّل أنك تُترجم فقرة كاملة من العربية إلى الإنجليزية، لكنك مُجبَر على قراءتها مرة واحدة، ثم إغلاق عينيك، وكتابة الترجمة بالكامل من الذاكرة دون العودة للنص الأصلي. هذا بالضبط ما كانت تفعله نماذج Seq2Seq القائمة على RNN/LSTM قبل عام 2014: يُضغَط النص المصدر بالكامل في **متجه واحد ثابت الحجم** (context vector) يُمرَّر للمُفكِّك (decoder)، بغض النظر عن طول الجملة — 5 كلمات أو 500 كلمة.

هذا "الاختناق" (bottleneck) كان السبب الرئيسي وراء ضعف أداء الترجمة الآلية في الجمل الطويلة. آلية **Attention** التي سنتعلمها في هذا الفصل حلّت هذه المشكلة بفكرة بسيطة وعبقرية: **اسمح للمُفكِّك بالنظر مباشرة إلى جميع حالات المُرمِّز (encoder) في كل خطوة توليد، مع إعطاء وزن مختلف لكل منها حسب أهميته**. هذه الفكرة — رغم أنها وُلِدت كترقيع لـ RNN — أصبحت لاحقًا جوهر Transformer بالكامل.

فهم Bahdanau وLuong Attention ليس مجرد تاريخ: فهو يمنحك الحدس الكامل لفهم لماذا صُمِّم Self-Attention بالضبط بالشكل الذي صُمِّم به (Query/Key/Value، أوزان الانتباه، التجميع الموزون).

---


## 2. الخريطة الذهنية


```text
مشكلة Seq2Seq التقليدي (2014)
│
├── Encoder RNN يقرأ الجملة المصدر بالكامل
├── يُنتج متجه سياق واحد ثابت الحجم h_T (آخر حالة مخفية)
├── Decoder RNN يستقبل h_T فقط لتوليد كل الكلمات
│
└── ❌ المشكلة: كل المعلومات مضغوطة في متجه واحد
    → أداء يتدهور بشدة مع طول الجملة (BLEU يهبط بعد 20 كلمة)

الحل: Attention (Bahdanau 2014 → Luong 2015)
│
├── احتفظ بكل الحالات المخفية للـ Encoder: h_1, h_2, ..., h_T
│   (وليس فقط الأخيرة)
│
├── في كل خطوة توليد t من الـ Decoder:
│   1. احسب "درجة توافق" (score/energy) بين حالة الـ Decoder الحالية
│      وكل حالة من حالات الـ Encoder
│   2. حوّل الدرجات إلى أوزان احتمالية عبر softmax → attention weights α
│   3. اجمع حالات الـ Encoder موزونة بـ α → context vector ديناميكي c_t
│   4. ادمج c_t مع حالة الـ Decoder لتوليد الكلمة التالية
│
└── ✅ النتيجة: كل كلمة مُولَّدة "تنظر" لأكثر أجزاء الجملة المصدر صلة بها
    (هذا بالضبط ما يفعله المترجم البشري: ينظر للكلمة المصدر المقابلة)

Bahdanau (Additive/Concat)          Luong (Multiplicative)
│                                    │
├── score = v^T tanh(W1·h_dec        ├── score = h_dec^T · h_enc (dot)
│           + W2·h_enc)              ├── score = h_dec^T · W · h_enc (general)
├── يُحسب قبل RNN step (input       ├── score = v^T tanh(W[h_dec;h_enc]) (concat)
│   feeding)                        ├── يُحسب بعد RNN step
└── أكثر تعبيرية لكن أبطأ          └── أبسط وأسرع حسابيًا
```


---


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل مترجمًا بشريًا يترجم الجملة: **"القطة الصغيرة الجميلة تنام على السجادة الحمراء"**. عندما يصل لترجمة كلمة "تنام" (sleeps)، فإن عينيه لا تنظر إلى كل الجملة بتساوٍ — بل تركّز بشكل أساسي على كلمة "تنام" ومحيطها القريب ("القطة")، بينما كلمة "الحمراء" في آخر الجملة لا تؤثر كثيرًا في هذه اللحظة بالذات.

هذا بالضبط ما توفره آلية Attention: **وزن ديناميكي متغير** يحدد "أين يجب أن أنظر" في كل خطوة من خطوات التوليد، بدلاً من الاعتماد على ذاكرة واحدة مضغوطة ثابتة.

### 🟡 طبقة الآلية: Bahdanau Attention (2014)

الصيغة الرياضية الكاملة لـ Bahdanau Attention (المعروفة أيضًا بـ Additive Attention):

**الخطوة 1 — حساب درجات التوافق (alignment scores):**

$$e_{t,i} = v_a^T \tanh(W_a h_{t-1}^{dec} + U_a h_i^{enc})$$

حيث:
- $h_{t-1}^{dec}$: الحالة المخفية للمفكك في الخطوة السابقة (قبل توليد الكلمة t)
- $h_i^{enc}$: الحالة المخفية للمرمز عند الموضع i
- $W_a, U_a, v_a$: أوزان قابلة للتعلم

**الخطوة 2 — تطبيع الدرجات إلى أوزان احتمالية:**

$$\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_{j=1}^{T} \exp(e_{t,j})}$$

**الخطوة 3 — حساب متجه السياق الديناميكي:**

$$c_t = \sum_{i=1}^{T} \alpha_{t,i} \cdot h_i^{enc}$$

**الخطوة 4 — دمج السياق مع حالة المفكك لتوليد الكلمة:**

$$\tilde{h}_t = \tanh(W_c[c_t ; h_{t-1}^{dec}])$$
$$y_t = \text{softmax}(W_o \tilde{h}_t)$$

ملاحظة مهمة: في Bahdanau الأصلي، يُحسب $c_t$ **قبل** تحديث حالة المفكك (input feeding)، أي أن $c_t$ يُدمَج مع تضمين الكلمة السابقة كمُدخل لخلية RNN في الخطوة t.

### 🟡 Luong Attention (2015) — تبسيط وتعميم

اقترح Luong ثلاث طرق لحساب الدرجات، وجعل حساب الانتباه **بعد** تحديث حالة المفكك (أبسط حسابيًا):

$$\text{score}(h_t^{dec}, h_i^{enc}) = \begin{cases} h_t^{dec \top} h_i^{enc} & \text{dot} \\ h_t^{dec \top} W_a h_i^{enc} & \text{general} \\ v_a^T \tanh(W_a[h_t^{dec}; h_i^{enc}]) & \text{concat} \end{cases}$$

ثم:
$$\alpha_t = \text{softmax}(\text{score}(h_t^{dec}, h^{enc}))$$
$$c_t = \sum_i \alpha_{t,i} h_i^{enc}$$
$$\tilde{h}_t = \tanh(W_c[c_t; h_t^{dec}])$$

### 🔴 طبقة التعمّق: الكود الكامل


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# 1. Bahdanau (Additive) Attention — من الصفر بـ PyTorch
# ============================================================
class BahdanauAttention(nn.Module):
    """
    آلية انتباه Bahdanau الإضافية (Additive/Concat Attention).
    تُستخدم عادة مع decoder RNN حيث تُحسب قبل خطوة RNN (input feeding).
    """
    def __init__(self, decoder_dim: int, encoder_dim: int, attn_dim: int = 256):
        super().__init__()
        self.W_a = nn.Linear(decoder_dim, attn_dim, bias=False)
        self.U_a = nn.Linear(encoder_dim, attn_dim, bias=False)
        self.v_a = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        """
        decoder_hidden:  (batch, decoder_dim)       حالة المفكك في t-1
        encoder_outputs: (batch, src_len, encoder_dim)  كل حالات المرمز
        mask: (batch, src_len)  1 للمواضع الحقيقية، 0 للـ padding
        """
        batch, src_len, _ = encoder_outputs.shape

        # وسّع حالة المفكك لتتوافق مع طول التسلسل المصدر
        dec_expanded = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)
        # (batch, src_len, decoder_dim)

        # e_{t,i} = v^T tanh(W_a h_dec + U_a h_enc)
        energy = torch.tanh(self.W_a(dec_expanded) + self.U_a(encoder_outputs))
        scores = self.v_a(energy).squeeze(-1)          # (batch, src_len)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)        # α_{t,i}

        # c_t = Σ α_{t,i} h_i^enc
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        context = context.squeeze(1)                    # (batch, encoder_dim)

        return context, attn_weights

# ============================================================
# 2. Luong (Multiplicative) Attention — بثلاث صيغ score
# ============================================================
class LuongAttention(nn.Module):
    def __init__(self, hidden_dim: int, method: str = "general"):
        super().__init__()
        assert method in ("dot", "general", "concat")
        self.method = method
        if method == "general":
            self.W_a = nn.Linear(hidden_dim, hidden_dim, bias=False)
        elif method == "concat":
            self.W_a = nn.Linear(hidden_dim * 2, hidden_dim, bias=False)
            self.v_a = nn.Linear(hidden_dim, 1, bias=False)

    def score(self, decoder_hidden, encoder_outputs):
        if self.method == "dot":
            # h_dec^T h_enc
            return torch.bmm(encoder_outputs, decoder_hidden.unsqueeze(2)).squeeze(2)
        elif self.method == "general":
            transformed = self.W_a(encoder_outputs)
            return torch.bmm(transformed, decoder_hidden.unsqueeze(2)).squeeze(2)
        else:  # concat
            src_len = encoder_outputs.size(1)
            dec_expanded = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)
            energy = torch.tanh(self.W_a(torch.cat([dec_expanded, encoder_outputs], dim=2)))
            return self.v_a(energy).squeeze(-1)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        scores = self.score(decoder_hidden, encoder_outputs)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attn_weights

# ============================================================
# 3. Seq2Seq كامل مع Attention (تطبيق للترجمة العربية-الإنجليزية)
# ============================================================
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.GRU(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, src, src_lengths):
        embedded = self.embedding(src)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_outputs, hidden = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)
        # دمج الاتجاهين الأمامي والخلفي لحالة البداية للمفكك
        hidden = torch.tanh(self.fc_hidden(torch.cat([hidden[0], hidden[1]], dim=1)))
        return outputs, hidden  # outputs: (batch, src_len, 2*hidden_dim)

class DecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, encoder_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = BahdanauAttention(hidden_dim, encoder_dim)
        self.rnn = nn.GRU(embed_dim + encoder_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim + encoder_dim + embed_dim, vocab_size)

    def forward(self, input_token, hidden, encoder_outputs, mask):
        embedded = self.embedding(input_token).unsqueeze(1)   # (batch, 1, embed_dim)

        context, attn_weights = self.attention(hidden, encoder_outputs, mask)
        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)

        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        output = output.squeeze(1)
        hidden = hidden.squeeze(0)

        prediction = self.fc_out(torch.cat([output, context, embedded.squeeze(1)], dim=1))
        return prediction, hidden, attn_weights


---


## 4. مثال محلول خطوة بخطوة

لنحسب يدويًا خطوة انتباه واحدة بأرقام مبسّطة.

**الجملة المصدر (عربية):** "القطة تنام" → بعد الترميز لدينا حالتان مخفيتان للمرمز (بأبعاد 2 للتبسيط):

$$h_1^{enc} = [1.0, 0.5] \quad (\text{"القطة"}), \qquad h_2^{enc} = [0.2, 0.9] \quad (\text{"تنام"})$$

**حالة المفكك الحالية** (قبل توليد كلمة "sleeps"):

$$h_{t-1}^{dec} = [0.3, 0.8]$$

**استخدام Luong Dot Attention** (الأبسط):

الخطوة 1 — حساب الدرجات (dot product):
$$e_1 = h_{t-1}^{dec} \cdot h_1^{enc} = (0.3)(1.0) + (0.8)(0.5) = 0.3 + 0.4 = 0.7$$
$$e_2 = h_{t-1}^{dec} \cdot h_2^{enc} = (0.3)(0.2) + (0.8)(0.9) = 0.06 + 0.72 = 0.78$$

الخطوة 2 — Softmax:
$$\alpha_1 = \frac{e^{0.7}}{e^{0.7}+e^{0.78}} = \frac{2.0138}{2.0138+2.1815} = \frac{2.0138}{4.1953} \approx 0.480$$
$$\alpha_2 = 1 - 0.480 = 0.520$$

الخطوة 3 — متجه السياق:
$$c_t = 0.480 \times [1.0, 0.5] + 0.520 \times [0.2, 0.9]$$
$$c_t = [0.480, 0.240] + [0.104, 0.468] = [0.584, 0.708]$$

**التفسير**: الوزنان متقاربان جدًا (0.48 و0.52) لأن حالة المفكك لم تحدد بعد تفضيلًا قويًا — لكن كلمة "تنام" حصلت على وزن أعلى قليلاً، وهو منطقي لأن المفكك على وشك توليد الفعل "sleeps" المقابل مباشرة لـ"تنام".

---


## 5. نصائح وحيل

> 💡 **استخدم الـ Masking دائمًا**: عند معالجة دفعات (batches) بجمل مختلفة الطول، يجب إخفاء مواضع الـ padding بقيمة `-inf` قبل softmax، وإلا ستتسرب "طاقة انتباه" مزيفة لرموز لا معنى لها.

> 💡 **تصوّر خرائط الانتباه (Attention Maps)**: ارسم مصفوفة $\alpha_{t,i}$ كخريطة حرارية؛ في الترجمة الجيدة سترى نمطًا شبه قطري يعكس التوافق بين الكلمات المصدر والهدف — أداة تشخيصية ممتازة.

> 💡 **Luong "general" غالبًا تفوز عمليًا**: أظهرت تجارب Luong الأصلية أن score العام (general) يُعطي أداءً أفضل من dot وconcat في معظم مهام الترجمة.

---


## 6. أخطاء شائعة ومصائد

> ⚠️ **الخلط بين توقيت الحساب**: في Bahdanau يُحسب $c_t$ **قبل** تحديث RNN (يُستخدم كمدخل)، بينما في Luong يُحسب **بعد** التحديث (يُدمج مع المخرج). الخلط بين الاثنين يُنتج تطبيقًا غير صحيح للورقتين الأصليتين.

> ⚠️ **نسيان تطبيع الدرجات بعد الإخفاء**: إذا طبّقت softmax قبل إخفاء الـ padding بدلاً من بعده، ستحصل على أوزان خاطئة تتضمن احتمالًا للرموز الفارغة.

> ⚠️ **الاعتقاد بأن Attention يلغي الحاجة لـ RNN بالكامل**: في هذا السياق التاريخي، Attention كانت **إضافة** لـ RNN وليست بديلاً عنها — البديل الكامل (Self-Attention بلا أي تكرار) لم يظهر إلا في Transformer عام 2017.

---


## 7. جدول مقارنة

| الخاصية | Bahdanau (2014) | Luong (2015) |
|---|---|---|
| **الاسم البديل** | Additive Attention | Multiplicative Attention |
| **دالة الدرجات** | $v^T\tanh(W_1h_{dec}+W_2h_{enc})$ | dot / general / concat |
| **توقيت الحساب** | قبل خطوة RNN (input feeding) | بعد خطوة RNN |
| **التعقيد الحسابي** | أعلى (طبقة tanh إضافية) | أقل (خصوصًا dot) |
| **اتجاه المرمز** | غالبًا ثنائي الاتجاه | أحادي أو ثنائي |
| **عدد المتغيرات** | صيغة واحدة | 3 صيغ score قابلة للاختيار |
| **الأداء التجريبي** | جيد جدًا، لكن أبطأ تدريبًا | غالبًا مماثل أو أفضل، وأسرع |
| **الأهمية التاريخية** | أول من قدّم الفكرة (ICLR 2015) | بسّطها وعمّمها (EMNLP 2015) |

---


## 8. ملخص أفضل الممارسات

- ✅ استخدم بالضرورة **Masking** عند التعامل مع دفعات بأطوال متفاوتة.
- ✅ فضّل Luong "general" أو "dot" كنقطة انطلاق بسيطة قبل تجربة concat الأثقل حسابيًا.
- ✅ تتبّع خرائط الانتباه أثناء التدريب كأداة تشخيص لصحة التقارب.
- ✅ تذكّر أن Attention في هذا الفصل لا يزال **مكمّلًا** لـ RNN وليس بديلًا — الفصل القادم سيُقدّم البديل الكامل.

---


## 9. أسئلة مقابلات

**س1: ما المشكلة الأساسية التي يحلّها Attention في نماذج Seq2Seq التقليدية؟**

**الإجابة النموذجية**: في Seq2Seq الكلاسيكي، يُضغَط التسلسل المصدر بالكامل — بغض النظر عن طوله — في متجه سياق واحد ثابت الحجم (آخر حالة مخفية للمرمز). هذا يخلق اختناق معلومات (information bottleneck): كلما طالت الجملة المصدر، زاد فقدان المعلومات المهمة، خصوصًا للكلمات البعيدة عن نهاية التسلسل. Attention يحلّ هذا بالسماح للمفكك بالوصول إلى **جميع** الحالات المخفية للمرمز في كل خطوة توليد، مع حساب وزن ديناميكي يحدد مدى أهمية كل موضع مصدر لحظة توليد كلمة هدف معينة، مما يُلغي الحاجة لضغط كل المعلومات في تمثيل واحد.

**س2: ما الفرق الجوهري بين دالة score في Bahdanau وLuong؟**

**الإجابة النموذجية**: Bahdanau يستخدم دالة إضافية (additive) غير خطية: يُركّب متجهي المفكك والمرمز عبر طبقتين خطيتين منفصلتين ثم جمعهما، يليهما tanh وطبقة خطية أخيرة لإنتاج رقم (score) واحد — هذا يمنح النموذج مرونة أكبر في تعلم علاقات معقدة بين المتجهين على حساب كلفة حسابية أعلى. Luong، في المقابل، يقترح صيغًا مضاعفة (multiplicative) أبسط — وأشهرها الضرب النقطي المباشر (dot) الذي لا يحتوي على أي معاملات قابلة للتعلم إطلاقًا — ما يجعله أسرع حسابيًا خصوصًا عند أبعاد كبيرة، لأنه يُختصر إلى عملية ضرب مصفوفات فعالة.

**س3: لماذا يُعتبر Bahdanau Attention نقطة انطلاق مفاهيمية لـ Self-Attention في Transformer رغم اختلافهما؟**

**الإجابة النموذجية**: كلاهما يتبع نفس النمط الثلاثي: (1) حساب درجة توافق/تشابه بين عنصر استعلام وعناصر مرجعية، (2) تطبيع هذه الدرجات عبر softmax لإنتاج أوزان احتمالية، (3) حساب مجموع موزون (context vector) لهذه العناصر المرجعية حسب الأوزان. الفرق الجوهري هو أن Bahdanau يحسب هذا **مرة واحدة فقط لكل خطوة توليد بين المفكك والمرمز** (cross-attention بين تسلسلين مختلفين معتمدَين على RNN)، بينما Self-Attention في Transformer يُطبّق هذا النمط **بين كل عنصر في التسلسل ونفسه** دون أي حاجة لـ RNN على الإطلاق، مما يتيح التوازي الكامل ويُزيل القيد التسلسلي بالكامل. فهم Bahdanau يمنحك حدس Query-Key-Value قبل رؤية الصيغة الرياضية المجردة لـ Transformer.

---


## 10. تمرين عملي

نفّذ الخطوات التالية:
1. اكتب جملتين عربيتين قصيرتين (3-4 كلمات لكل منهما) وامثّل كل كلمة بمتجه عشوائي ثنائي البعد.
2. طبّق Luong Dot Attention يدويًا (بالقلم والورقة أو بكود NumPy بسيط) لحساب أوزان الانتباه عندما تكون حالة المفكك مساوية لمتجه كلمة معينة من الجملة الأولى.
3. غيّر صيغة score إلى "general" بإضافة مصفوفة وزن عشوائية $W_a$ وقارن الفرق في توزيع الأوزان الناتجة.
4. ارسم خريطة حرارية (heatmap) لمصفوفة الانتباه الكاملة إذا كانت الحالة المفكك متعددة الخطوات.

---


## 11. مراجع للتعمّق

- Bahdanau, D., Cho, K., & Bengio, Y. (2015). *Neural Machine Translation by Jointly Learning to Align and Translate*. ICLR.
- Luong, M. T., Pham, H., & Manning, C. D. (2015). *Effective Approaches to Attention-based Neural Machine Translation*. EMNLP.
- Jurafsky & Martin, *Speech and Language Processing*, الفصل الخاص بـ Attention في Seq2Seq.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — المحوّلات الأساسية / الفصل 9: من التكرار إلى الانتباه: آلية Attention التاريخية.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
